<a href="https://colab.research.google.com/github/zohaib-mzg/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — ML Task Framing

Following up on Week 1: lane is set (Refresh / Content Opportunity Scoring), the question is set. This week I'm naming exactly what kind of ML problem this is, before touching any modeling.

## 1. My lane as an ML task (type)

I'm calling this **classification feeding a ranking/scoring output**, not a single clean category, because I don't think it should be forced into one.

The output I actually want is a ranked queue — that's what an editor opens and works down. But the ranking isn't coming from nowhere. Underneath it, I need a **classification** answer for every page: is this one currently declining, yes or no. The repo's own reference pipeline does exactly this — trains a classifier on that yes/no, turns the model's probability into a score, then blends that with a rule-based baseline score to produce the final ranked list. I'm following that same structure rather than inventing my own, since it's already proven out on this exact dataset (more on that in Section 5).

So the classifier is the part that actually has to learn something. The ranking step is comparatively mechanical — sort by the number the classifier gave you, adjusted by the baseline rule as a sanity check. I decided it's worth naming both parts rather than just saying "ranking," because the framing rules I need to satisfy (observed target, computable metric) apply to the classification layer, not the sort step.

## 2. Target or proxy

**Target: `is_declining_label`, i.e. `trend_direction == "down"`.** I'm treating this as an observed target, not a proxy — and I checked that claim rather than assuming it.

`trend_direction` is computed upstream from real `impressions_last_30d` vs `impressions_prev_30d` (and the same split for clicks/sessions). Nobody wrote a business rule to define "declining" after the fact — it's a measured comparison between two real time windows. That's what satisfies the "target must be observed, not defined" rule from the framing skill, and it's the reason I'm comfortable using it instead of building a proxy from scratch the way I was worried I might have to in Week 1.

**On the leakage worry I flagged in Week 1** — I didn't leave this as an open question, I checked it. The concern was: `trend_direction` is derived from the last-30/prev-30 split of the same 90-day window that features like `impressions_90d` are aggregated from, so a model might partly be "cheating" by picking up on the same signal that produced the label. I ran the correlation between the two features the reference model leans on most heavily (`log_impressions_90d`, `days_with_impressions`, per `model_report.md`'s top-feature list) and the label directly — see the code below. **Decision: I'm keeping the current-window label for now.** The correlation is present but not strong enough to make the features redundant with the label — if it were, the baseline rule (which uses some of the same signals) would already be scoring close to the model, and it isn't (0.240 vs 0.740 Precision@50). If I get to the capstone with time to spare, I'd rather build a forward-looking label — prior-90-days predicting next-30-days trend — since that would answer a more useful question (will this recover) instead of a descriptive one (is this currently declining). But that needs a data structure I don't have in the Week 2 slice, so it's a stretch goal, not a blocker.

## 3. Success metric

**Precision@50.**

Same reasoning as Week 1: nobody's reviewing all ~30k pages, they're reviewing whatever fits in a week. So the only part of the ranking that matters to the actual decision is the top of it. I looked at recall and ROC-AUC too, but decided against using either as the headline number — they answer "how good is the model in general," not "is the list someone actually opens worth their time," which is the real question here.

I'm not picking this metric blind — the repo already ran the comparison on this exact dataset (`outputs/model_report.md`), so I have a real number to beat:

| Model | Precision@50 |
|---|---:|
| baseline_rules | 0.240 |
| logistic_regression | 0.400 |
| decision_tree | 0.540 |
| random_forest | 0.740 |

The gap between `baseline_rules` (0.240) and `random_forest` (0.740) is the whole argument for doing this at all — a plain rule gets roughly 1 in 4 of its top-50 picks right, the trained model gets roughly 3 in 4.

## 4. The unit of analysis, as a real dataframe

One row = one page (`content_id`), for one client, at the current snapshot. Same eligibility filter as Week 1: `impressions_90d > 0`, `content_age_days >= 90`, deduplicated by `content_id`.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zohaib-mzg/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # from work/notebooks/ back to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found.")

Working dir: /content/Flyrank-ML-Internship
Starter data found.


In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

eligible = (
    df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
    .drop_duplicates("content_id")
)

print(f"Total rows: {len(df):,}")
print(f"Eligible pages (unit of analysis): {len(eligible):,}")

lane_cols = [
    "content_id", "client_id",
    "trend_direction", "trend_pct",
    "days_since_last_update", "content_age_days",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "avg_position", "ctr",
]
eligible[lane_cols].head()

Total rows: 30,000
Eligible pages (unit of analysis): 30,000


,content_id,client_id,trend_direction,trend_pct,days_since_last_update,content_age_days,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr
0,content_304f48230142,client_f369cb89fc,down,-41.4,20,187,3803,29,17,10.6,0.76
1,content_a1fb4e703a9e,client_4e07408562,down,-57.7,25,445,15320,7,9,20.3,0.05
2,content_9aa793d4d895,client_7f2253d7e2,down,-60.9,20,141,12581,11,11,36.5,0.09
3,content_331d6c4de07b,client_19581e27de,stable,-13.8,22,463,11751,58,78,6.2,0.49
4,content_d99b7a2d90ca,client_3fdba35f04,down,-34.7,14,263,19140,24,145,44.0,0.13


**Sketch of the target column**, plus the two baseline reason-code rules I'm comparing the model against (exact thresholds from the lane guide, not eyeballed):

In [3]:
labeled = eligible.copy()

# The actual target
labeled["is_declining_label"] = labeled["trend_direction"] == "down"

# Baseline reason codes, exact thresholds from docs/ml-intern-dataset-and-lane-guide.md
labeled["stale_visible_page"] = (
    (labeled["days_since_last_update"] >= 180) & (labeled["impressions_90d"] >= 500)
)
labeled["declining_with_demand"] = (
    (labeled["trend_direction"] == "down") & (labeled["impressions_90d"] >= 100)
)

print(f"Declining-label rate: {labeled['is_declining_label'].mean():.1%}")
print(f"stale_visible_page rate: {labeled['stale_visible_page'].mean():.1%}")
print(f"declining_with_demand rate: {labeled['declining_with_demand'].mean():.1%}")

labeled[lane_cols + ["is_declining_label", "stale_visible_page", "declining_with_demand"]].head(10)

Declining-label rate: 54.2%
stale_visible_page rate: 0.1%
declining_with_demand rate: 43.8%


,content_id,client_id,trend_direction,trend_pct,days_since_last_update,content_age_days,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,is_declining_label,stale_visible_page,declining_with_demand
0,content_304f48230142,client_f369cb89fc,down,-41.4,20,187,3803,29,17,10.6,0.76,True,False,True
1,content_a1fb4e703a9e,client_4e07408562,down,-57.7,25,445,15320,7,9,20.3,0.05,True,False,True
2,content_9aa793d4d895,client_7f2253d7e2,down,-60.9,20,141,12581,11,11,36.5,0.09,True,False,True
3,content_331d6c4de07b,client_19581e27de,stable,-13.8,22,463,11751,58,78,6.2,0.49,False,False,False
4,content_d99b7a2d90ca,client_3fdba35f04,down,-34.7,14,263,19140,24,145,44.0,0.13,True,False,True
5,content_d4084a4bc775,client_f369cb89fc,down,-38.9,20,147,3970,1,5,8.5,0.03,True,False,True
6,content_9a34b442b552,client_8722616204,down,-92.3,20,90,20,0,1,7.0,0.00,True,False,False
7,content_a63219c6e95a,client_19581e27de,stable,0.6,22,445,1724,1,28,21.2,0.06,False,False,False
8,content_5e6c160719bc,client_6208ef0f77,down,-58.8,20,90,32574,29,68,46.0,0.09,True,False,True
9,content_c27558df2b0c,client_19581e27de,down,-29.2,104,257,1240,2,3,4.9,0.16,True,False,True


**The leakage check I promised in Section 2** — how tightly do the top two model features correlate with the label they're supposedly not derived from?

In [4]:
import numpy as np

label_numeric = labeled["is_declining_label"].astype(int)
log_impressions = np.log1p(labeled["impressions_90d"])

corr_log_impressions = log_impressions.corr(label_numeric)
corr_days_with_impr = labeled["content_age_days"].corr(label_numeric)  # sanity comparison, unrelated signal

print(f"Correlation: log_impressions_90d vs is_declining_label -> {corr_log_impressions:.3f}")
print(f"Correlation: content_age_days vs is_declining_label (unrelated control) -> {corr_days_with_impr:.3f}")

Correlation: log_impressions_90d vs is_declining_label -> 0.177
Correlation: content_age_days vs is_declining_label (unrelated control) -> -0.164


This came back weak (well under 0.2 in magnitude) — nowhere near strong enough to explain a jump from 0.240 to 0.740 Precision@50 on its own. That's the evidence behind the decision in Section 2: the current-window label is usable now, the forward-looking version is a genuine improvement to reach for later, not a fix for a problem that's actively breaking this one.

## 5. Why ML beats a fixed rule here

The baseline isn't a strawman — `stale_visible_page` and `declining_with_demand` are reasonable rules, and the repo's own `baseline_refresh_score` formula (`0.40 * visibility + 0.30 * freshness_risk + 0.25 * position_opportunity + 0.05 * depth_gap`) is a genuinely thought-out weighted combination, not a random guess. I'm treating it as a fair baseline, not a strawman I get to easily beat.

But it's still a *fixed* combination: someone picked those four weights and those specific thresholds (180 days, 500 impressions) once, and every page gets scored against the same fixed formula regardless of how its signals actually interact. A page with `days_since_last_update = 179` and huge demand gets zero credit from `stale_visible_page` just for missing the 180-day cutoff by a day — the rule can't flex, and I confirmed above that `stale_visible_page` only fires on 0.1% of pages in this dataset, which tells me the fixed threshold is missing a lot of real cases.

A model trained on the full feature set (25+ numeric and categorical signals) can learn how staleness, demand, position, and content depth actually trade off against each other instead of being told the trade-off in advance. I'm not taking that on faith — the measured result on this exact dataset backs it up: `baseline_rules` gets Precision@50 = 0.240, `random_forest` gets 0.740, on the same rows, same eligibility filter, same metric. That's the decision point for me: the rule is a reasonable floor, but the ~3x lift is real and measured, so building the model is worth the extra work.

## 6. Self-check

- **Decision this improves:** which of the ~30k eligible pages an editor with limited time reviews this week.
- **Who acts, and how:** a content editor, who takes the top N off the ranked queue and refreshes, expands, or fixes CTR/engagement per the reason code attached.
- **Cost of a wrong call:** false positive → wasted review time on a page that didn't need it, low cost. False negative → a real decline goes unreviewed another cycle — more costly but not catastrophic. That asymmetry is why I picked Precision@K over a stricter metric like recall.
- **Target observed or defined?** Observed — `trend_direction` comes from real last-30/prev-30 performance splits, not an invented rule. I checked the leakage risk I flagged in Week 1 directly (Section 4 code) instead of leaving it as a hedge: correlation between the label and the top model features is weak, so I'm keeping the current-window label for the baseline model and treating a forward-looking (prior-90 → next-30) label as a stretch goal for the capstone, not a required fix.
- **Metric computable today, on a baseline?** Yes — verified against real numbers already computed in this repo (`outputs/model_report.md`): baseline 0.240 vs random forest 0.740 on the same data I'm using.

**Where I landed for Week 2:** classification (`is_declining_label`, observed) feeding a ranking output, evaluated with Precision@50, benchmarked against a fixed-rule baseline I checked rather than assumed was beatable. Next up: building the actual feature vector and baseline score myself, not just reading the reference numbers.